# MAIZ_14D · Segunda ronda de revisión · GPU L4

**Entorno: GPU L4.** Pulse **Ejecutar todas**. Tarda alrededor de 1 h. Si se corta la luz o la sesión, vuelva a pulsar **Ejecutar todas**: lo ya calculado se conserva en Drive y no se repite.

Este notebook responde dos pedidos del Revisor 2 en la segunda ronda:

1. **Comentario M5 (f).** Búsqueda de imágenes parecidas con DINOv2 **entre cada cohorte externa y las imágenes internas de entrenamiento y validación**. En MAIZ_14A la búsqueda se hizo solo dentro de cada cohorte externa.
2. **Comentario M10 (ii).** Controles de fuente con las **fuentes primarias reales** (Tanzania, PlantVillage, Adege, CCMT) en lugar de los seis componentes operativos. Reentrena el clasificador de fuente con la receta archivada (MobileNetV3-Large, semilla 17, hasta 12 épocas).

No entrena ni modifica ningún modelo de enfermedades o plagas.

**Resultado:** `MAIZ_14D_Resultados.zip`, en `REVISION_POST_REVIEW_20261029/` y como descarga. Envíe ese ZIP al chat.

In [ ]:
import shutil,subprocess
if not shutil.which('nvidia-smi'):raise RuntimeError('Este archivo requiere GPU. Selecciónela en Entorno de ejecución > Cambiar tipo.')
p=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True,timeout=15)
print('GPU:',p.stdout.strip())

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/TesisQ1_Maiz')
if not ROOT.is_dir() or not (ROOT/'01_datasets').is_dir():
    raise FileNotFoundError('No se encuentra TesisQ1_Maiz/01_datasets en Mi unidad.')
REV=ROOT/'REVISION_POST_REVIEW_20261029'
need=['training_data_preflight.json','training_data_cache','evaluation_5seeds/prepared/internal_test/images.tar',
      'evaluation_5seeds/prepared/pandian2019/images.tar','evaluation_5seeds/prepared/plantvillage/images.tar',
      'evaluation_5seeds/prepared/tom2024_primary/images.tar','MAIZ14_cierre_parciales/dinov2_embeddings.npz']
falta=[n for n in need if not (REV/n).exists()]
for n in need:print(('OK   ' if n not in falta else 'FALTA'),n)
if [n for n in falta if 'dinov2' not in n]:raise FileNotFoundError('Faltan archivos necesarios: '+', '.join(falta))

In [ ]:
# Paquetes: no se reinstala torch. timm se fija en la versión usada para entrenar.
import importlib.metadata as md, subprocess, sys
def ver(p):
    try:return md.version(p)
    except md.PackageNotFoundError:return None
pk=[]
if ver('timm')!='1.0.15':pk.append('timm==1.0.15')
if ver('ImageHash') is None:pk.append('imagehash==4.3.1')
if pk:subprocess.run([sys.executable,'-m','pip','install','-q',*pk],check=True)
import torch,torchvision,timm
print('torch',torch.__version__,'| torchvision',torchvision.__version__,'| timm',timm.__version__)

In [ ]:
# Código de MAIZ_14D (verificado por SHA-256)
import base64,hashlib,io,zipfile,importlib,sys
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='fd87b6efa9438dc5799548774d2df57d46965b06147884318e8edd9dd0b4406d'
CODE=Path('/content/MAIZ14_code');CODE.mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(raw)).extractall(CODE)
sys.path.insert(0,str(CODE))
import maiz14,maiz14_run as R,maiz14d as D
importlib.reload(maiz14);importlib.reload(R);importlib.reload(D)
print('Versión:',maiz14.VERSION,'+14d | archivos:',sorted(p.name for p in CODE.iterdir()))

In [ ]:
# Ejecución completa (reanudable)
D.run_all(ROOT)

In [ ]:
import pandas as pd
from IPython.display import display, Image as IPImage
O=ROOT/D.OUT_REL;pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
print('1) Controles sin imagen: seis componentes frente a cuatro fuentes primarias')
display(pd.read_csv(O/'source_only_controls_components_vs_primary_sources.csv').T)
print('2) Clasificador de fuente con fuentes primarias')
display(pd.read_csv(O/'source_classifier_primary_sources.csv'))
print('3) DINOv2: imágenes externas con un vecino interno parecido')
display(pd.read_csv(O/'dino_external_vs_internal_summary.csv').T)
display(pd.read_csv(O/'dino_internal_reference.csv'))

In [ ]:
# Hojas de contacto: los 24 pares externa–interna más parecidos de cada cohorte (izquierda: externa; derecha: interna)
for p in sorted((O/'contact_sheets').glob('*.png')):
    print(p.name);display(IPImage(filename=str(p),width=1100))

In [ ]:
from google.colab import files
files.download('/content/MAIZ_14D_Resultados.zip')